# This is done in google collab because the local device had computational limitations

In [1]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

cd /content/drive/MyDrive/Colab Notebooks/

/content/drive/MyDrive/Colab Notebooks


In [3]:
from transformers import AutoImageProcessor, AutoModel as ImageModel
from transformers import AutoTokenizer, AutoModel as TextModel
import torch
from PIL import Image

In [4]:
# ── RAD-DINO: radiology-specific image encoder (Microsoft) ──────────────
image_processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")
image_model = ImageModel.from_pretrained("microsoft/rad-dino")

# ── PubMedBERT: biomedical text encoder (Microsoft) ──────────────────────
text_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)
text_model = TextModel.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
image_model.to(device)
text_model.to(device)

print(f"Using device: {device}")
print(f"RAD-DINO and PubMedBERT loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda
RAD-DINO and PubMedBERT loaded successfully.


In [5]:
def get_text_embedding(text):
    """
    Generate text embedding using PubMedBERT.
    Uses mean pooling over last hidden states (masked to ignore padding).
    Output dim: 768
    """
    inputs = text_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512  # PubMedBERT supports up to 512 tokens
    ).to(device)

    with torch.no_grad():
        outputs = text_model(**inputs)

    # Mean pooling — mask out padding tokens before averaging
    attention_mask = inputs["attention_mask"]
    token_embeddings = outputs.last_hidden_state  # (1, seq_len, 768)
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    emb = torch.sum(token_embeddings * mask_expanded, dim=1) / \
          torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

    # L2 normalise
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu()

In [6]:
def get_image_embedding(image_path):
    """
    Generate image embedding using RAD-DINO.
    Extracts the [CLS] token from the last hidden state.
    Output dim: 768
    """
    image = Image.open(image_path).convert("RGB")

    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = image_model(**inputs)

    # RAD-DINO is DINOv2-based: use the [CLS] token (index 0) of last hidden state
    emb = outputs.last_hidden_state[:, 0, :]  # shape: (1, 768)

    # L2 normalise
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu()

In [7]:
import os
def list_image_files(directory):
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp']
    image_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if any(file.lower().endswith(ext) for ext in image_extensions):
                image_files.append(os.path.join(root, file))
    return image_files

In [8]:
drive_path = '/content/drive/MyDrive/Colab Notebooks/images/'
all_image_files = list_image_files(drive_path)

if all_image_files:
    print("Found image files:")
    for img_file in all_image_files:
        print(img_file)
else:
    print("No image files found in Google Drive.")

Found image files:
/content/drive/MyDrive/Colab Notebooks/images/3653_IM-1815-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3660_IM-1820-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3663_IM-1822-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3679_IM-1831-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3694_IM-1845-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3677_IM-1830-2001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3684_IM-1836-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/367_IM-1826-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3675_IM-1829-0001-0001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3699_IM-1846-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3697_IM-1846-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3692_IM-1843-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/images/3680_IM-1832-1001.dcm.png
/content/drive/MyDrive/Colab Notebooks/im

In [9]:
if all_image_files:
    image_to_use = all_image_files[0]  # Using the first found image as an example
    print(f"Using image: {image_to_use}")
    image_embedding = get_image_embedding(image_to_use)
    print(f"Length of image embedding: {len(image_embedding)}")
else:
    print("No image files found to process.")

Using image: /content/drive/MyDrive/Colab Notebooks/images/3653_IM-1815-1001.dcm.png
Length of image embedding: 768


In [10]:
import json
data = None
with open ('./embedding_input.json','r') as f:
  data = json.load(f)
print(data[0])

{'id': 1, 'text': 'Indication: Positive TB test | Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no of a pleural effusion. There is no evidence of pneumothorax. | Impression: Normal chest', 'image': '1_IM-0001-4001.dcm.png', 'label': 'Pulmonary Disease', 'cluster': 1, 'normal': True}


In [11]:
from collections import Counter
uids = [item['id'] for item in data]
# Count occurrences of each UID
uid_counts = Counter(uids)
# Identify repeats
repeated_uids = {uid: count for uid, count in uid_counts.items() if count > 1}
# Results summary
print(f"Total entries in JSON: {len(data)}")
print(f"Total unique UIDs: {len(uid_counts)}")
print(f"Number of repeated UIDs: {len(repeated_uids)}")
if repeated_uids:
    print("\nRepeated UIDs and their counts:")
    for uid, count in repeated_uids.items():
        print(f"UID {uid}: {count} times")
else:
    print("\nSuccess: No repeated UIDs found! dataset is unique.")

Total entries in JSON: 3666
Total unique UIDs: 3666
Number of repeated UIDs: 0

Success: No repeated UIDs found! dataset is unique.


In [12]:
import numpy as np

results_to_save = []
n = len(data)
count = 0
print(f"Processing {n} entries...")
for entry in data:
    # Get text embedding
    text_data = entry['text']
    text_embedding = get_text_embedding(text_data)

    # Get image embedding
    image_name = entry['image']
    image_path = os.path.join(drive_path, image_name)

    try:
        image_embedding = get_image_embedding(image_path)

        # Calculate final embedding (weighted average)
        # final_embedding = 0.6 * image_embedding + 0.4 * text_embedding

        # Store as lists for JSON serialization
        results_to_save.append({
            "id": entry['id'],
            "text_embedding": text_embedding.tolist(),
            "cleaned_text":entry['text'],
            "image_embedding": image_embedding.tolist(),
            "image":entry['image'],
            'label':entry['label'],
            'cluster':entry['cluster'],
            'normal':entry['normal']
        })
        count += 1
        print(f'processed {count} out of {n} files, file id : {entry['id']}')
    except Exception as e:
        print(f"Error processing ID {entry['id']}: {e}")



Processing 3666 entries...
processed 1 out of 3666 files, file id : 1
processed 2 out of 3666 files, file id : 2
processed 3 out of 3666 files, file id : 3
processed 4 out of 3666 files, file id : 4
processed 5 out of 3666 files, file id : 5
processed 6 out of 3666 files, file id : 6
processed 7 out of 3666 files, file id : 7
processed 8 out of 3666 files, file id : 8
processed 9 out of 3666 files, file id : 9
processed 10 out of 3666 files, file id : 10
processed 11 out of 3666 files, file id : 11
processed 12 out of 3666 files, file id : 12
processed 13 out of 3666 files, file id : 13
processed 14 out of 3666 files, file id : 14
processed 15 out of 3666 files, file id : 15
processed 16 out of 3666 files, file id : 17
processed 17 out of 3666 files, file id : 18
processed 18 out of 3666 files, file id : 19
processed 19 out of 3666 files, file id : 20
processed 20 out of 3666 files, file id : 21
processed 21 out of 3666 files, file id : 22
processed 22 out of 3666 files, file id : 23
p

In [13]:
# Save to JSON file
output_file_path = '/content/drive/MyDrive/Colab Notebooks/embedding_results.json'
with open(output_file_path, 'w') as f:
    json.dump(results_to_save, f)

print(f"Successfully saved {len(results_to_save)} entries to {output_file_path}")

Successfully saved 3666 entries to /content/drive/MyDrive/Colab Notebooks/embeddings_results.json


In [14]:
check = []
for entry in results_to_save:
  check.append(entry['id'])

from collections import Counter
# find all id with count more than one
duplicates = [item for item, count in Counter(check).items() if count > 1]
print(duplicates)

[]
